In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("PriceCast — Modelagem de Preços")
print("Bibliotecas carregadas com sucesso!")


PriceCast — Modelagem de Preços
Bibliotecas carregadas com sucesso!


In [2]:

# Carregar a base tratada
df_precos = pd.read_csv(
    "df_precos.csv",
    encoding="utf-8-sig"
)

# Restaurar o tipo de data
df_precos["InvoiceDate"] = pd.to_datetime(
    df_precos["InvoiceDate"]
)

# Conferir a importação
print("Base carregada com sucesso!")
print("Dimensões:", df_precos.shape)

display(df_precos.head())


Base carregada com sucesso!
Dimensões: (502514, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,AnoMes
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12


In [3]:

# 1. Conferir a estrutura da base
print("Dimensões:", df_precos.shape)

# 2. Conferir os tipos de dados
print("\nTipos das variáveis:")
display(df_precos.dtypes)

# 3. Conferir o período dos registros
print("\nData inicial:", df_precos["InvoiceDate"].min())
print("Data final:", df_precos["InvoiceDate"].max())

# 4. Validar critérios básicos
print("\nValores ausentes por coluna:")
display(df_precos.isna().sum())

print("\nPreços menores ou iguais a zero:",
      (df_precos["Price"] <= 0).sum())

print("Quantidades menores ou iguais a zero:",
      (df_precos["Quantity"] <= 0).sum())

print("Duplicatas exatas:", df_precos.duplicated().sum())


Dimensões: (502514, 9)

Tipos das variáveis:


Invoice                 int64
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
AnoMes                    str
dtype: object


Data inicial: 2009-12-01 07:45:00
Data final: 2010-12-09 20:01:00

Valores ausentes por coluna:


Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    102916
Country             0
AnoMes              0
dtype: int64


Preços menores ou iguais a zero: 0
Quantidades menores ou iguais a zero: 0
Duplicatas exatas: 0


Vamos começar com um modelo de regressão que prevê Price, o preço unitário registrado na transação.
A divisão será cronológica:
- Treino: registros anteriores a outubro de 2010.
- Teste: registros de outubro de 2010 em diante.
Assim, simulamos uma situação mais realista: treinar com dados históricos e avaliar o desempenho em um período posterior.

In [4]:

# Garantir ordenação cronológica
df_modelo = df_precos.sort_values("InvoiceDate").copy()

# Separar treino e teste por data
data_corte = pd.Timestamp("2010-10-01")

treino = df_modelo[df_modelo["InvoiceDate"] < data_corte].copy()
teste = df_modelo[df_modelo["InvoiceDate"] >= data_corte].copy()

# Definir a variável-alvo
y_treino = treino["Price"]
y_teste = teste["Price"]

print("Data de corte:", data_corte)
print("Registros de treino:", len(treino))
print("Registros de teste:", len(teste))

print("\nPeríodo de treino:")
print(treino["InvoiceDate"].min(), "até", treino["InvoiceDate"].max())

print("\nPeríodo de teste:")
print(teste["InvoiceDate"].min(), "até", teste["InvoiceDate"].max())

print("\nProporção do teste: {:.2f}%".format(
    len(teste) / len(df_modelo) * 100
))


Data de corte: 2010-10-01 00:00:00
Registros de treino: 349414
Registros de teste: 153100

Período de treino:
2009-12-01 07:45:00 até 2010-09-30 20:08:00

Período de teste:
2010-10-01 08:19:00 até 2010-12-09 20:01:00

Proporção do teste: 30.47%


Essa estratégia é adequada para uma previsão com dimensão temporal, pois evita misturar aleatoriamente transações antigas e futuras entre treino e teste.
Um cuidado importante: dezembro de 2010 está incompleto na base, terminando em 9 de dezembro. Portanto, o período de teste não representa um trimestre completo.
2. Próxima etapa: preparar as variáveis
Antes de treinar, vamos definir quais informações o modelo poderá usar para estimar Price.
Para começar, usaremos:
- StockCode: código do produto.
- Country: país da transação.
- Quantity: quantidade comprada, caso esteja disponível no momento da estimativa.
- InvoiceDate: vamos extrair mês, dia da semana e dia do mês.
Não usaremos Invoice nem Customer ID como variáveis preditoras, pois são identificadores de transação e cliente. Também não usaremos Description nesta primeira versão, para manter o modelo inicial mais simples.

In [5]:

# Criar cópias para preservar a base original
treino_modelo = treino.copy()
teste_modelo = teste.copy()

# Extrair variáveis temporais
for base in [treino_modelo, teste_modelo]:
    base["Mes"] = base["InvoiceDate"].dt.month
    base["DiaMes"] = base["InvoiceDate"].dt.day
    base["DiaSemana"] = base["InvoiceDate"].dt.dayofweek

# Definir as variáveis preditoras
features = [
    "StockCode",
    "Country",
    "Quantity",
    "Mes",
    "DiaMes",
    "DiaSemana"
]

X_treino = treino_modelo[features]
X_teste = teste_modelo[features]

y_treino = treino_modelo["Price"]
y_teste = teste_modelo["Price"]

print("Dimensões de X_treino:", X_treino.shape)
print("Dimensões de X_teste:", X_teste.shape)

print("\nVariáveis preditoras:")
print(X_treino.columns.tolist())

print("\nValores ausentes nas variáveis preditoras:")
print(X_treino.isnull().sum())


Dimensões de X_treino: (349414, 6)
Dimensões de X_teste: (153100, 6)

Variáveis preditoras:
['StockCode', 'Country', 'Quantity', 'Mes', 'DiaMes', 'DiaSemana']

Valores ausentes nas variáveis preditoras:
StockCode    0
Country      0
Quantity     0
Mes          0
DiaMes       0
DiaSemana    0
dtype: int64


## Pré-processamento e modelo
Vamos começar com a Regressão Ridge, um modelo de regressão regularizada. É uma boa primeira referência porque permite treinar com variáveis categóricas codificadas e avaliar o desempenho antes de testar modelos mais complexos.

In [6]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

# Variáveis categóricas e numéricas
colunas_categoricas = ["StockCode", "Country"]
colunas_numericas = [
    "Quantity",
    "Mes",
    "DiaMes",
    "DiaSemana"
]

# Pré-processamento das variáveis categóricas
preprocessamento_categorico = OneHotEncoder(
    handle_unknown="ignore",
    min_frequency=10
)

# Pré-processamento das variáveis numéricas
preprocessamento_numerico = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("escala", StandardScaler())
])

# Combinar os pré-processamentos
preprocessador = ColumnTransformer([
    (
        "categoricas",
        preprocessamento_categorico,
        colunas_categoricas
    ),
    (
        "numericas",
        preprocessamento_numerico,
        colunas_numericas
    )
])

# Criar o pipeline completo
modelo_ridge = Pipeline([
    ("preprocessamento", preprocessador),
    ("regressor", Ridge(alpha=1.0))
])

print("Pipeline criado com sucesso!")
print(modelo_ridge)


Pipeline criado com sucesso!
Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(transformers=[('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=10),
                                                  ['StockCode', 'Country']),
                                                 ('numericas',
                                                  Pipeline(steps=[('imputacao',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('escala',
                                                                   StandardScaler())]),
                                                  ['Quantity', 'Mes', 'DiaMes',
                                                   'DiaSemana'])])),
                ('regressor', Ridge())])


In [9]:

# Validar a base tratada e a divisão dos dados

print("BASE TRATADA")
print(f"Registros: {len(df_precos):,}")
print(f"Colunas: {df_precos.shape[1]}")
print(f"Duplicatas: {df_precos.duplicated().sum()}")
print(f"Preços inválidos: {(df_precos['Price'] <= 0).sum()}")
print(f"Quantidades inválidas: {(df_precos['Quantity'] <= 0).sum()}")

print("\nDIVISÃO DOS DADOS")
print(f"Treino: {len(X_treino):,} registros")
print(f"Teste: {len(X_teste):,} registros")

print("\nPRIMEIROS REGISTROS DA BASE TRATADA")
display(df_precos.head(10))


BASE TRATADA
Registros: 502,514
Colunas: 9
Duplicatas: 0
Preços inválidos: 0
Quantidades inválidas: 0

DIVISÃO DOS DADOS
Treino: 349,414 registros
Teste: 153,100 registros

PRIMEIROS REGISTROS DA BASE TRATADA


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,AnoMes
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2009-12
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2009-12
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,2009-12
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,2009-12


In [10]:

print("=== VALIDAÇÃO DA BASE TRATADA ===")
print(f"Total de registros: {len(df_precos):,}")
print(f"Total de colunas: {df_precos.shape[1]}")
print(f"Duplicatas: {df_precos.duplicated().sum()}")
print(f"Preços menores ou iguais a zero: {(df_precos['Price'] <= 0).sum()}")
print(f"Quantidades menores ou iguais a zero: {(df_precos['Quantity'] <= 0).sum()}")

print("\n=== DIVISÃO TEMPORAL ===")
print(f"Treino: {len(X_treino):,} registros")
print(f"Teste: {len(X_teste):,} registros")
print(f"Data inicial do treino: {treino['InvoiceDate'].min()}")
print(f"Data final do treino: {treino['InvoiceDate'].max()}")
print(f"Data inicial do teste: {teste['InvoiceDate'].min()}")
print(f"Data final do teste: {teste['InvoiceDate'].max()}")


=== VALIDAÇÃO DA BASE TRATADA ===
Total de registros: 502,514
Total de colunas: 9
Duplicatas: 0
Preços menores ou iguais a zero: 0
Quantidades menores ou iguais a zero: 0

=== DIVISÃO TEMPORAL ===
Treino: 349,414 registros
Teste: 153,100 registros
Data inicial do treino: 2009-12-01 07:45:00
Data final do treino: 2010-09-30 20:08:00
Data inicial do teste: 2010-10-01 08:19:00
Data final do teste: 2010-12-09 20:01:00


A divisão temporal também está correta: o treinamento utiliza dados até setembro de 2010, enquanto o teste começa em outubro de 2010. Isso permite avaliar o modelo em um período posterior.

In [11]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import time

# Iniciar cronômetro
inicio = time.time()

# Treinar o modelo
modelo_ridge.fit(X_treino, y_treino)

# Prever preços no conjunto de teste
y_pred = modelo_ridge.predict(X_teste)

# Calcular métricas
mae = mean_absolute_error(y_teste, y_pred)
rmse = np.sqrt(mean_squared_error(y_teste, y_pred))
r2 = r2_score(y_teste, y_pred)

tempo = time.time() - inicio

# Exibir resultados
print("PRICECAST — REGRESSÃO RIDGE")
print("=" * 40)
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")
print(f"Tempo de execução: {tempo:.2f} segundos")


PRICECAST — REGRESSÃO RIDGE
MAE: 0.9997
RMSE: 2.7539
R²: 0.6027
Tempo de execução: 3.19 segundos


O que esses números significam?
- MAE = 0,9997: em média, a diferença absoluta entre o preço real e o previsto foi de aproximadamente uma unidade monetária.
- RMSE = 2,7539: os erros maiores influenciam consideravelmente o resultado. Isso é compatível com a presença de preços extremos na base.
- R² = 0,6027: o modelo explica aproximadamente 60,27% da variação dos preços no conjunto de teste. Ainda existe espaço para melhorar.
Esses resultados são um ponto de partida, não uma prova de que o modelo está comercialmente pronto. Precisamos compará-lo com uma referência simples e com outros algoritmos. Como a base tem preços extremos, também será importante examinar os erros por faixa de preço.

# Próxima etapa — Analisar os erros do Ridge

In [12]:

# Criar tabela para analisar as previsões
resultado_ridge = teste[[
    "StockCode", "Description", "Price", "Quantity", "InvoiceDate"
]].copy()

resultado_ridge["Preco_Previsto"] = y_pred
resultado_ridge["Erro_Absoluto"] = (
    resultado_ridge["Price"] - resultado_ridge["Preco_Previsto"]
).abs()

resultado_ridge["Erro"] = (
    resultado_ridge["Price"] - resultado_ridge["Preco_Previsto"]
)

# Conferir previsões individuais
display(
    resultado_ridge[
        [
            "StockCode",
            "Description",
            "Price",
            "Preco_Previsto",
            "Erro_Absoluto"
        ]
    ].head(15).round(2)
)

print("\nErro absoluto médio:", round(
    resultado_ridge["Erro_Absoluto"].mean(), 4
))

print("Mediana do erro absoluto:", round(
    resultado_ridge["Erro_Absoluto"].median(), 4
))


,StockCode,Description,Price,Preco_Previsto,Erro_Absoluto
349414,82494L,WOODEN FRAME ANTIQUE WHITE,2.95,3.09,0.14
349415,22423,REGENCY CAKESTAND 3 TIER,12.75,14.54,1.79
349425,21327,SKULLS WRITING SET,1.65,1.34,0.31
349424,21988,PACK OF 6 SKULL PAPER PLATES,0.85,0.30,0.55
349423,22553,PLASTERS IN TIN SKULLS,1.65,1.20,0.45
349422,71406C,BLACK ORANGE SQUEEZER,0.42,4.34,3.92
349421,22327,ROUND SNACK BOXES SET OF 4 SKULLS,2.95,2.77,0.18
349420,22316,200 BENDY SKULL STRAWS,1.25,0.82,0.43
349418,35599B,BLACK CHRISTMAS TREE 120CM,8.49,4.35,4.14
349417,35598B,BLACK CHRISTMAS TREE 60CM,2.95,2.81,0.14



Erro absoluto médio: 0.9997
Mediana do erro absoluto: 0.4654


In [19]:

# Evitar divisão por zero
resultado_ridge["Erro_Percentual"] = np.where(
    resultado_ridge["Price"] != 0,
    resultado_ridge["Erro_Absoluto"]
    / resultado_ridge["Price"].abs() * 100,
    np.nan
)

display(
    resultado_ridge[
        [
            "StockCode",
            "Description",
            "Price",
            "Preco_Previsto",
            "Erro_Absoluto",
            "Erro_Percentual"
        ]
    ].head(15).round(2)
)

print(
    "Erro percentual absoluto mediano (%):",
    round(resultado_ridge["Erro_Percentual"].median(), 2)
)



,StockCode,Description,Price,Preco_Previsto,Erro_Absoluto,Erro_Percentual
349414,82494L,WOODEN FRAME ANTIQUE WHITE,2.95,3.09,0.14,4.76
349415,22423,REGENCY CAKESTAND 3 TIER,12.75,14.54,1.79,14.07
349425,21327,SKULLS WRITING SET,1.65,1.34,0.31,18.63
349424,21988,PACK OF 6 SKULL PAPER PLATES,0.85,0.30,0.55,64.89
349423,22553,PLASTERS IN TIN SKULLS,1.65,1.20,0.45,27.15
349422,71406C,BLACK ORANGE SQUEEZER,0.42,4.34,3.92,933.44
349421,22327,ROUND SNACK BOXES SET OF 4 SKULLS,2.95,2.77,0.18,6.08
349420,22316,200 BENDY SKULL STRAWS,1.25,0.82,0.43,34.53
349418,35599B,BLACK CHRISTMAS TREE 120CM,8.49,4.35,4.14,48.82
349417,35598B,BLACK CHRISTMAS TREE 60CM,2.95,2.81,0.14,4.76


Erro percentual absoluto mediano (%): 23.43


In [20]:

# Recriar a tabela com os erros do modelo Ridge
resultado_ridge["Preco_Previsto"] = y_pred

resultado_ridge["Erro_Absoluto"] = (
    resultado_ridge["Price"] - resultado_ridge["Preco_Previsto"]
).abs()

# Ordenar pelos maiores erros reais
maiores_erros = resultado_ridge.nlargest(
    20, "Erro_Absoluto"
)

display(
    maiores_erros[
        [
            "StockCode",
            "Description",
            "Price",
            "Preco_Previsto",
            "Erro_Absoluto"
        ]
    ].round(2)
)


,StockCode,Description,Price,Preco_Previsto,Erro_Absoluto
405371,22827,RUSTIC SEVENTEEN DRAWER SIDEBOARD,165.0,3.71,161.29
390340,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.0,3.89,161.11
452634,22827,RUSTIC SEVENTEEN DRAWER SIDEBOARD,165.0,3.91,161.09
388331,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.0,3.99,161.01
400901,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.0,3.99,161.01
411671,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.0,4.08,160.92
481136,22827,RUSTIC SEVENTEEN DRAWER SIDEBOARD,165.0,4.09,160.91
445987,22828,REGENCY MIRROR WITH SHUTTERS,165.0,4.09,160.91
499591,22828,REGENCY MIRROR WITH SHUTTERS,165.0,4.10,160.90
410378,22827,RUSTIC SEVENTEEN DRAWER SIDEBOARD,165.0,4.17,160.83


Os três produtos que você já estava investigando — 22827, 22828 e 22823 — dominam a lista dos maiores erros. Isso sugere um problema específico na capacidade do modelo de prever produtos caros, e precisamos verificar se esses códigos ficaram fora do treino.

In [17]:

codigos = ["22827", "22828", "22823"]

for codigo in codigos:
    historico = df_precos[
        df_precos["StockCode"].astype(str) == codigo
    ]

    print(f"\nProduto: {codigo}")
    print(f"Registros totais: {len(historico)}")
    print(f"Preço mínimo: {historico['Price'].min():.2f}")
    print(f"Preço mediano: {historico['Price'].median():.2f}")
    print(f"Preço máximo: {historico['Price'].max():.2f}")

    historico_treino = treino[
        treino["StockCode"].astype(str) == codigo
    ]

    print(f"Registros no treino: {len(historico_treino)}")
    if len(historico_treino) > 0:
        print(
            "Preço mediano no treino:",
            round(historico_treino["Price"].median(), 2)
        )



Produto: 22827
Registros totais: 11
Preço mínimo: 145.00
Preço mediano: 165.00
Preço máximo: 165.00
Registros no treino: 0

Produto: 22828
Registros totais: 5
Preço mínimo: 145.00
Preço mediano: 145.00
Preço máximo: 165.00
Registros no treino: 0

Produto: 22823
Registros totais: 5
Preço mínimo: 110.00
Preço mediano: 110.00
Preço máximo: 125.00
Registros no treino: 0


Como usamos uma divisão temporal, esses produtos surgiram depois da data de corte de 1º de outubro de 2010. O modelo não teve oportunidade de aprender seus preços históricos. O OneHotEncoder com handle_unknown="ignore" consegue processar esses códigos, mas não tem informação específica sobre eles.
Próximo passo: medir o problema de produtos novos
Como o objetivo do PriceCast é estimar preços de produtos que já fazem parte do catálogo, precisamos distinguir duas situações:
- Produtos conhecidos: aparecem no treino e no teste.
- Produtos não vistos: aparecem no teste, mas não no treino.
Vamos medir quantos registros de teste pertencem a cada grupo.

In [21]:

print("MODELO:", type(modelo_ridge).__name__)

print("\nFORMATO DOS DADOS")
print("X_treino:", X_treino.shape)
print("X_teste:", X_teste.shape)

print("\nCOLUNAS DE ENTRADA")

if hasattr(X_treino, "columns"):
    print(list(X_treino.columns))
else:
    print("X_treino é uma matriz, sem nomes de colunas.")

print("\nCONFIGURAÇÃO DO MODELO")

if hasattr(modelo_ridge, "get_params"):
    print(modelo_ridge.get_params())

print("\nEXEMPLO DE ENTRADAS DO TREINO")
display(X_treino.head() if hasattr(X_treino, "head")
        else X_treino[:5])


MODELO: Pipeline

FORMATO DOS DADOS
X_treino: (349414, 6)
X_teste: (153100, 6)

COLUNAS DE ENTRADA
['StockCode', 'Country', 'Quantity', 'Mes', 'DiaMes', 'DiaSemana']

CONFIGURAÇÃO DO MODELO
{'memory': None, 'steps': [('preprocessamento', ColumnTransformer(transformers=[('categoricas',
                                 OneHotEncoder(handle_unknown='ignore',
                                               min_frequency=10),
                                 ['StockCode', 'Country']),
                                ('numericas',
                                 Pipeline(steps=[('imputacao',
                                                  SimpleImputer(strategy='median')),
                                                 ('escala', StandardScaler())]),
                                 ['Quantity', 'Mes', 'DiaMes', 'DiaSemana'])])), ('regressor', Ridge())], 'transform_input': None, 'verbose': False, 'preprocessamento': ColumnTransformer(transformers=[('categoricas',
                        

,StockCode,Country,Quantity,Mes,DiaMes,DiaSemana
0,85048,United Kingdom,12,12,1,1
1,79323P,United Kingdom,12,12,1,1
2,79323W,United Kingdom,12,12,1,1
3,22041,United Kingdom,48,12,1,1
4,21232,United Kingdom,24,12,1,1


Os produtos 22827, 22828 e 22823 não aparecem no conjunto de treinamento, segundo sua análise anterior.
O Ridge não conhece o preço desses produtos. Como não temos a descrição do produto entre as entradas mostradas, o modelo depende das demais variáveis e não consegue identificar que se trata de móveis de alto valor.
Importante: não significa necessariamente que o Ridge esteja mal implementado. Significa que talvez as variáveis atuais não contenham informação suficiente para prever preços de produtos nunca vistos.

In [22]:

# Verificar como as variáveis categóricas são processadas

print("MODELO:", modelo_ridge)

print("\nTIPOS DAS COLUNAS DE ENTRADA:")
print(X_treino.dtypes if hasattr(X_treino, "dtypes") else "Matriz sem nomes de colunas")

print("\nCÓDIGOS DOS PRODUTOS:")
print("Produtos distintos no treino:", treino["StockCode"].nunique())
print("Produtos distintos no teste:", teste["StockCode"].nunique())

print("\nDESCRIÇÃO DOS PRODUTOS COM MAIOR ERRO:")
codigos = ["22827", "22828", "22823"]

for codigo in codigos:
    historico = df_precos[
        df_precos["StockCode"].astype(str) == codigo
    ]

    colunas = [
        c for c in ["StockCode", "Description", "Price"]
        if c in historico.columns
    ]

    print(f"\nProduto: {codigo}")
    display(historico[colunas].drop_duplicates().head(10))


MODELO: Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(transformers=[('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=10),
                                                  ['StockCode', 'Country']),
                                                 ('numericas',
                                                  Pipeline(steps=[('imputacao',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('escala',
                                                                   StandardScaler())]),
                                                  ['Quantity', 'Mes', 'DiaMes',
                                                   'DiaSemana'])])),
                ('regressor', Ridge())])

TIPOS DAS COLUNAS DE ENTRADA:
StockCode 

,StockCode,Description,Price
388331,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.0
388688,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,145.0
405371,22827,RUSTIC SEVENTEEN DRAWER SIDEBOARD,165.0
499592,22827,RUSTIC SEVENTEEN DRAWER SIDEBOARD,145.0



Produto: 22828


,StockCode,Description,Price
388964,22828,REGENCY MIRROR WITH SHUTTERS,145.0
445987,22828,REGENCY MIRROR WITH SHUTTERS,165.0



Produto: 22823


,StockCode,Description,Price
396123,22823,CHEST NATURAL WOOD 20 DRAWERS,110.0
400998,22823,CHEST NATURAL WOOD 20 DRAWERS,125.0


In [23]:

print("COLUNAS DA BASE:")
print(df_precos.columns.tolist())

print("\nCOLUNAS DO TREINO:")
print(X_treino.columns.tolist())

print("\nFORMATO DO TREINO E TESTE:")
print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

print("\nMODELO ATUAL:")
print(modelo_ridge)


COLUNAS DA BASE:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'AnoMes']

COLUNAS DO TREINO:
['StockCode', 'Country', 'Quantity', 'Mes', 'DiaMes', 'DiaSemana']

FORMATO DO TREINO E TESTE:
Treino: (349414, 6)
Teste: (153100, 6)

MODELO ATUAL:
Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(transformers=[('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=10),
                                                  ['StockCode', 'Country']),
                                                 ('numericas',
                                                  Pipeline(steps=[('imputacao',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('escala',
                             

temos o diagnóstico necessário para avançar com o PriceCast.
Seu modelo atual usa OneHotEncoder para StockCode e Country, além de variáveis numéricas. Como o codificador está configurado com handle_unknown='ignore', um código de produto ausente no treinamento é ignorado na codificação. Assim, o Ridge não consegue aprender diretamente o preço daquele produto pelo seu código.
Também descobrimos que a coluna Description existe na base original, mas não está sendo utilizada no modelo.
O que vamos mudar
Vamos criar uma segunda versão do PriceCast, preservando o Ridge atual para comparação.
- Modelo atual: código do produto, país, quantidade e data.
- Novo modelo: descrição textual com TF-IDF, país, quantidade e data, inicialmente sem StockCode.
- Avaliação: MAE, RMSE e R², além de um teste específico com produtos que não aparecem no treinamento.
Retirar StockCode da nova versão é importante: queremos verificar se o modelo consegue generalizar a partir das características descritas no texto, em vez de depender de um código que nunca viu.
Próximo passo: treinar o modelo com TF-IDF
Execute a célula abaixo no notebook. Ela cria um novo pipeline sem substituir modelo_ridge.

In [24]:

import pandas as pd
import numpy as np
import time

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Preparar as entradas sem usar StockCode
def preparar_dados_texto(df):
    dados = df.copy()

    # Garantir que as variáveis de data existam
    if not all(c in dados.columns for c in ["Mes", "DiaMes", "DiaSemana"]):
        dados["InvoiceDate"] = pd.to_datetime(
            dados["InvoiceDate"], errors="coerce"
        )
        dados["Mes"] = dados["InvoiceDate"].dt.month
        dados["DiaMes"] = dados["InvoiceDate"].dt.day
        dados["DiaSemana"] = dados["InvoiceDate"].dt.dayofweek

    dados["Description"] = (
        dados["Description"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    return dados[
        ["Description", "Country", "Quantity",
         "Mes", "DiaMes", "DiaSemana"]
    ]

X_treino_texto = preparar_dados_texto(treino)
X_teste_texto = preparar_dados_texto(teste)

y_treino_texto = treino["Price"]
y_teste_texto = teste["Price"]

# 2. Transformar texto e demais variáveis
preprocessador_texto = ColumnTransformer(
    transformers=[
        (
            "texto",
            TfidfVectorizer(
                max_features=20000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True
            ),
            "Description"
        ),
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=10
            ),
            ["Country"]
        ),
        (
            "numericas",
            Pipeline([
                ("imputacao", SimpleImputer(strategy="median")),
                ("escala", StandardScaler())
            ]),
            ["Quantity", "Mes", "DiaMes", "DiaSemana"]
        )
    ]
)

# 3. Criar e treinar o novo modelo
modelo_ridge_texto = Pipeline([
    ("preprocessamento", preprocessador_texto),
    ("regressor", Ridge())
])

inicio = time.time()

modelo_ridge_texto.fit(X_treino_texto, y_treino_texto)
predicoes_texto = modelo_ridge_texto.predict(X_teste_texto)

tempo_texto = time.time() - inicio

# 4. Avaliar
mae_texto = mean_absolute_error(y_teste_texto, predicoes_texto)
rmse_texto = np.sqrt(
    mean_squared_error(y_teste_texto, predicoes_texto)
)
r2_texto = r2_score(y_teste_texto, predicoes_texto)

print("PRICECAST — RIDGE + TF-IDF")
print(f"MAE: {mae_texto:.4f}")
print(f"RMSE: {rmse_texto:.4f}")
print(f"R²: {r2_texto:.4f}")
print(f"Tempo: {tempo_texto:.2f} segundos")


PRICECAST — RIDGE + TF-IDF
MAE: 0.9590
RMSE: 2.6163
R²: 0.6414
Tempo: 17.29 segundos


O modelo explica aproximadamente 64,14% da variação dos preços no conjunto de teste. O custo foi aumentar o tempo de execução, algo esperado ao processar descrições textuais.

In [25]:

codigos = ["22827", "22828", "22823"]

for codigo in codigos:
    mascara = teste["StockCode"].astype(str) == codigo

    if not mascara.any():
        print(f"\nProduto {codigo}: não encontrado no teste.")
        continue

    # Usar exatamente as mesmas linhas e entradas do modelo novo
    X_produto = X_teste_texto.loc[mascara]
    y_real = teste.loc[mascara, "Price"]
    y_previsto = modelo_ridge_texto.predict(X_produto)

    print(f"\nProduto: {codigo}")
    print(f"Descrição: {X_produto['Description'].iloc[0]}")
    print(f"Registros no teste: {mascara.sum()}")
    print(f"Preço real mediano: £{y_real.median():.2f}")
    print(f"Preço previsto mediano: £{np.median(y_previsto):.2f}")
    print(f"Erro absoluto mediano: £{np.median(np.abs(y_real - y_previsto)):.2f}")



Produto: 22827
Descrição: GIANT SEVENTEEN DRAWER SIDEBOARD
Registros no teste: 11
Preço real mediano: £165.00
Preço previsto mediano: £3.02
Erro absoluto mediano: £161.89

Produto: 22828
Descrição: REGENCY MIRROR WITH SHUTTERS
Registros no teste: 5
Preço real mediano: £145.00
Preço previsto mediano: £7.39
Erro absoluto mediano: £138.15

Produto: 22823
Descrição: CHEST NATURAL WOOD 20 DRAWERS
Registros no teste: 5
Preço real mediano: £110.00
Preço previsto mediano: £1.70
Erro absoluto mediano: £108.75


O modelo continua prevendo preços muito baixos. Portanto, não devemos considerar o problema resolvido só porque o R² aumentou de 0,6027 para 0,6414.
A descrição ajuda um pouco em alguns casos, mas palavras como SIDEBOARD, MIRROR e CHEST não fornecem, sozinhas, informação suficiente para estimar o valor comercial desses itens.
Há também uma hipótese importante: a base pode conter uma distribuição de preços muito concentrada em produtos baratos, fazendo o modelo regredir para valores baixos quando encontra uma combinação de características inédita.

In [26]:

print("=== DISTRIBUIÇÃO DOS PREÇOS NO TREINO ===")
display(y_treino_texto.describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
).to_frame("Preço"))

print("\n=== QUANTIDADE DE REGISTROS POR FAIXA DE PREÇO ===")
faixas = pd.cut(
    y_treino_texto,
    bins=[-np.inf, 5, 20, 50, 100, 200, np.inf],
    labels=[
        "Até £5", "£5–£20", "£20–£50",
        "£50–£100", "£100–£200", "Acima de £200"
    ]
)
display(faixas.value_counts(sort=False).to_frame("Registros"))

print("\n=== PRODUTOS CAROS NO TREINO ===")
caros = (
    treino.loc[treino["Price"] >= 100,
               ["StockCode", "Description", "Price"]]
    .drop_duplicates()
    .sort_values("Price", ascending=False)
)
display(caros.head(20))

print("\n=== PROPORÇÃO DE PREÇOS ACIMA DE £100 ===")
print(f"Treino: {(y_treino_texto >= 100).mean():.4%}")
print(f"Teste: {(y_teste_texto >= 100).mean():.4%}")


=== DISTRIBUIÇÃO DOS PREÇOS NO TREINO ===


,Preço
count,349414.000000
mean,3.468585
std,5.401859
min,0.001000
50%,2.100000
75%,4.210000
90%,7.620000
95%,9.960500
99%,16.980000
99.9%,39.950000



=== QUANTIDADE DE REGISTROS POR FAIXA DE PREÇO ===


,Registros
Price,
Até £5,282535
£5–£20,64262
£20–£50,2460
£50–£100,107
£100–£200,18
Acima de £200,32



=== PRODUTOS CAROS NO TREINO ===


,StockCode,Description,Price
183578,84016,FLAG OF ST GEORGE CAR FLAG,1157.15
130107,84016,FLAG OF ST GEORGE CAR FLAG,867.79
172950,84016,FLAG OF ST GEORGE CAR FLAG,408.40
256138,22656,VINTAGE BLUE KITCHEN CABINET,295.00
256139,22655,VINTAGE RED KITCHEN CABINET,295.00
108957,84016,FLAG OF ST GEORGE CAR FLAG,280.75
237249,84016,FLAG OF ST GEORGE CAR FLAG,272.27
330878,84965B,BLUE KASHMIRI COFFEE TABLE,129.95
101773,21760,FRENCH STYLE WALL DRESSER,125.00
189991,85070,ANT WHITE SWEETHEART TABLE W 3 DRAW,125.00



=== PROPORÇÃO DE PREÇOS ACIMA DE £100 ===
Treino: 0.0143%
Teste: 0.0176%


In [27]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

alvos = ["22827", "22828", "22823"]

base = df_precos.copy()
base["StockCode"] = base["StockCode"].astype(str)
base["Description"] = (
    base["Description"].fillna("").astype(str).str.upper().str.strip()
)

# Usamos apenas registros que pertencem ao conjunto de treinamento
codigos_treino = set(treino["StockCode"].astype(str))

produtos_teste = (
    base[base["StockCode"].isin(alvos)]
    [["StockCode", "Description", "Price"]]
    .drop_duplicates(subset=["StockCode", "Description"])
)

produtos_treino_caros = (
    base[
        base["StockCode"].isin(codigos_treino)
        & (base["Price"] >= 50)
        & (~base["StockCode"].isin(alvos))
    ][["StockCode", "Description", "Price"]]
    .drop_duplicates(subset=["StockCode", "Description"])
    .reset_index(drop=True)
)

print("Produtos caros disponíveis para comparação:",
      len(produtos_treino_caros))

if len(produtos_treino_caros) == 0:
    print("Não há produtos para comparar.")
else:
    textos = (
        produtos_teste["Description"].tolist()
        + produtos_treino_caros["Description"].tolist()
    )

    tfidf = TfidfVectorizer(ngram_range=(1, 2))
    matriz = tfidf.fit_transform(textos)

    n = len(produtos_teste)
    similaridades = cosine_similarity(matriz[:n], matriz[n:])

    for i, (_, produto) in enumerate(produtos_teste.iterrows()):
        melhores = similaridades[i].argsort()[::-1][:3]

        print(
            f"\nProduto {produto['StockCode']}: "
            f"{produto['Description']}"
        )

        display(
            produtos_treino_caros.iloc[melhores][
                ["StockCode", "Description", "Price"]
            ].assign(
                Similaridade=similaridades[i][melhores]
            ).reset_index(drop=True)
        )


Produtos caros disponíveis para comparação: 34

Produto 22827: GIANT SEVENTEEN DRAWER SIDEBOARD


,StockCode,Description,Price,Similaridade
0,22503,CABIN BAG VINTAGE PAISLEY,59.53,0.0
1,22655,VINTAGE RED KITCHEN CABINET,295.00,0.0
2,22656,VINTAGE BLUE KITCHEN CABINET,295.00,0.0



Produto 22828: REGENCY MIRROR WITH SHUTTERS


,StockCode,Description,Price,Similaridade
0,21759,METAL WALL SHELF UNIT WITH HOOKS,102.13,0.095737
1,22503,CABIN BAG VINTAGE PAISLEY,59.53,0.000000
2,22655,VINTAGE RED KITCHEN CABINET,295.00,0.000000



Produto 22823: CHEST NATURAL WOOD 20 DRAWERS


,StockCode,Description,Price,Similaridade
0,22503,CABIN BAG VINTAGE PAISLEY,59.53,0.0
1,22655,VINTAGE RED KITCHEN CABINET,295.00,0.0
2,22656,VINTAGE BLUE KITCHEN CABINET,295.00,0.0



Produto 22827: RUSTIC  SEVENTEEN DRAWER SIDEBOARD


,StockCode,Description,Price,Similaridade
0,22503,CABIN BAG VINTAGE PAISLEY,59.53,0.0
1,22655,VINTAGE RED KITCHEN CABINET,295.00,0.0
2,22656,VINTAGE BLUE KITCHEN CABINET,295.00,0.0


## Vamos mudar a estratégia do PriceCast
Já aprendemos três coisas:
- O Ridge original tem bom desempenho geral, mas falha em produtos inéditos de alto valor.
- O TF-IDF melhorou as métricas gerais, mas não resolveu esses casos.
- Os produtos caros são raros no treinamento, o que dificulta a generalização.
Minha recomendação agora é parar de fazer buscas manuais isoladas e realizar um teste controlado de generalização. Vamos comparar os modelos em produtos inteiros que ficam fora do treinamento, e não apenas em uma divisão aleatória de linhas.

In [28]:

print("TREINO")
print("Linhas:", len(treino))
print("Produtos distintos:", treino["StockCode"].nunique())

print("\nTESTE")
print("Linhas:", len(teste))
print("Produtos distintos:", teste["StockCode"].nunique())

codigos_treino = set(treino["StockCode"].astype(str))
codigos_teste = set(teste["StockCode"].astype(str))

print("\nCÓDIGOS DE PRODUTOS")
print("Presentes nos dois conjuntos:", len(codigos_treino & codigos_teste))
print("Exclusivos do teste:", len(codigos_teste - codigos_treino))
print("Exclusivos do treino:", len(codigos_treino - codigos_teste))


TREINO
Linhas: 349414
Produtos distintos: 4060

TESTE
Linhas: 153100
Produtos distintos: 3271

CÓDIGOS DE PRODUTOS
Presentes nos dois conjuntos: 3088
Exclusivos do teste: 183
Exclusivos do treino: 972


Isso significa que seu teste mistura dois cenários diferentes:
- Produtos conhecidos: 3.088 códigos aparecem tanto no treino quanto no teste.
- Produtos inéditos: 183 códigos aparecem somente no teste.
Portanto, as métricas gerais do Ridge e do Ridge + TF-IDF combinam esses dois cenários. Um bom resultado geral pode esconder erros graves nos produtos inéditos — exatamente o que observamos.
A próxima etapa é medir o desempenho nesses 183 produtos separadamente. Assim saberemos se o problema está concentrado nos produtos nunca vistos.

In [29]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Identificar os produtos que o modelo nunca viu no treino
codigos_treino = set(treino["StockCode"].astype(str))
codigos_teste = teste["StockCode"].astype(str)

mascara_ineditos = ~codigos_teste.isin(codigos_treino)
mascara_conhecidos = ~mascara_ineditos

# Previsões dos dois modelos nos mesmos registros de teste
pred_ridge_original = modelo_ridge.predict(X_teste)
pred_ridge_texto = modelo_ridge_texto.predict(X_teste_texto)

y_real = teste["Price"].to_numpy()

def avaliar(nome, mascara, y, pred):
    if mascara.sum() == 0:
        return {
            "Modelo": nome,
            "Grupo": "Sem registros",
            "Registros": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan
        }

    # R² não é definido para um único registro
    r2 = (
        r2_score(y[mascara], pred[mascara])
        if mascara.sum() >= 2 else np.nan
    )

    return {
        "Modelo": nome,
        "Grupo": "",
        "Registros": int(mascara.sum()),
        "MAE": mean_absolute_error(y[mascara], pred[mascara]),
        "RMSE": np.sqrt(
            mean_squared_error(y[mascara], pred[mascara])
        ),
        "R2": r2
    }

resultados = []

for nome, pred in [
    ("Ridge original", pred_ridge_original),
    ("Ridge + TF-IDF", pred_ridge_texto)
]:
    for grupo, mascara in [
        ("Todos os produtos", np.ones(len(teste), dtype=bool)),
        ("Produtos conhecidos", mascara_conhecidos.to_numpy()),
        ("Produtos inéditos", mascara_ineditos.to_numpy())
    ]:
        resultado = avaliar(nome, mascara, y_real, pred)
        resultado["Grupo"] = grupo
        resultados.append(resultado)

comparacao = pd.DataFrame(resultados)

display(
    comparacao.round(4)
)


,Modelo,Grupo,Registros,MAE,RMSE,R2
0,Ridge original,Todos os produtos,153100,0.9997,2.7539,0.6027
1,Ridge original,Produtos conhecidos,145990,0.8137,1.6257,0.8171
2,Ridge original,Produtos inéditos,7110,4.8180,10.4422,-0.1048
3,Ridge + TF-IDF,Todos os produtos,153100,0.9590,2.6163,0.6414
4,Ridge + TF-IDF,Produtos conhecidos,145990,0.7962,1.4818,0.8480
5,Ridge + TF-IDF,Produtos inéditos,7110,4.3036,10.1149,-0.0366


O TF-IDF reduziu o MAE dos produtos inéditos em aproximadamente 10,7%. Isso é uma melhora real, embora insuficiente para resolver o problema.
O R² negativo indica que, nesse grupo, o modelo tem desempenho inferior ao de uma previsão constante baseada na média dos preços reais desse mesmo grupo. Isso não significa que todas as previsões sejam ruins, mas mostra que ainda há muito espaço para melhorar.

O que faremos agora?
Minha recomendação é testar um modelo de referência baseado no preço mediano por produto, para os produtos conhecidos, e manter o Ridge + TF-IDF como candidato para os inéditos. Entretanto, para produtos realmente novos, o preço mediano histórico não existe; por isso, precisamos de uma estratégia separada.
Antes de construir essa abordagem, vamos verificar se os 183 produtos inéditos são, de fato, responsáveis pelos 7.110 registros de teste e como seus preços se distribuem.

In [30]:

# Avaliar os produtos inéditos como grupo
teste_analise = teste.copy()
teste_analise["StockCode"] = teste_analise["StockCode"].astype(str)

codigos_treino = set(treino["StockCode"].astype(str))

teste_analise["GrupoProduto"] = np.where(
    teste_analise["StockCode"].isin(codigos_treino),
    "Conhecido",
    "Inédito"
)

print("RESUMO DOS PREÇOS POR GRUPO")
display(
    teste_analise.groupby("GrupoProduto")["Price"]
    .agg(
        Registros="count",
        Produtos="nunique",
        Media="mean",
        Mediana="median",
        Minimo="min",
        Maximo="max"
    )
    .round(3)
)

print("\nPRODUTOS INÉDITOS COM MAIOR PREÇO MEDIANO")
display(
    teste_analise[teste_analise["GrupoProduto"] == "Inédito"]
    .groupby(["StockCode", "Description"], dropna=False)["Price"]
    .agg(["count", "median", "max"])
    .sort_values("median", ascending=False)
    .head(15)
)


RESUMO DOS PREÇOS POR GRUPO


,Registros,Produtos,Media,Mediana,Minimo,Maximo
GrupoProduto,,,,,,
Conhecido,145990,254,3.151,2.10,0.001,295.0
Inédito,7110,106,7.213,4.95,0.360,165.0



PRODUTOS INÉDITOS COM MAIOR PREÇO MEDIANO


count  median     max
StockCode Description                                              
22827     GIANT SEVENTEEN DRAWER SIDEBOARD        6  165.00  165.00
          RUSTIC  SEVENTEEN DRAWER SIDEBOARD      5  165.00  165.00
22828     REGENCY MIRROR WITH SHUTTERS            5  145.00  165.00
22823     CHEST NATURAL WOOD 20 DRAWERS           5  110.00  125.00
22929     SCHOOL DESK AND CHAIR                   4   65.00   65.00
22833     HALL CABINET WITH 3 DRAWERS            10   49.95   49.95
22824     3 TIER SWEETHEART GARDEN SHELF          7   35.95   35.95
22803     IVORY EMBROIDERED QUILT                14   35.75   39.95
22830     UTILTY CABINET WITH HOOKS              16   26.95   55.28
84968f    SET OF 16 VINTAGE SKY BLUE CUTLERY      1   25.49   25.49
22769     CHALKBOARD KITCHEN ORGANISER           39   24.95   51.02
22782     SET 3 WICKER STORAGE BASKETS           10   24.95   24.95
22764     RUSTIC WOODEN CABINET, GLASS DOORS     22   24.95   24.95
22761     CHEST 7 DRAWER MA CAMPAGNE             11   24.95   24.95
22783     SET 3 WICKER OVAL BASKETS W LIDS       16   19.95   19.95

Agora temos uma evidência importante para o PriceCast: a dificuldade com produtos inéditos está concentrada, em parte, em produtos de decoração e mobiliário com preços elevados.

## Próximo experimento: comparar preço real e preço previsto

In [42]:

# Comparar preços reais e previstos nos produtos inéditos

avaliacao = teste.copy().reset_index(drop=True)

avaliacao["StockCode"] = avaliacao["StockCode"].astype(str)

# Gerar previsões para as mesmas linhas do conjunto de teste
avaliacao["Preco_Previsto"] = modelo_ridge_texto.predict(X_teste_texto)

avaliacao["Erro_Absoluto"] = (
    avaliacao["Price"] - avaliacao["Preco_Previsto"]
).abs()

# Identificar produtos que não apareceram no treino
codigos_treino = set(treino["StockCode"].astype(str))

avaliacao["GrupoProduto"] = np.where(
    avaliacao["StockCode"].isin(codigos_treino),
    "Conhecido",
    "Inédito"
)

# Resumir os 15 produtos inéditos mais caros
resumo_erros = (
    avaliacao[avaliacao["GrupoProduto"] == "Inédito"]
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        Registros=("Price", "count"),
        Preco_Real_Mediano=("Price", "median"),
        Preco_Previsto_Mediano=("Preco_Previsto", "median"),
        Erro_Absoluto_Medio=("Erro_Absoluto", "mean")
    )
    .sort_values("Preco_Real_Mediano", ascending=False)
)

display(resumo_erros.head(15).round(2))


Registros  Preco_Real_Mediano  \
StockCode Description                                                         
22827     GIANT SEVENTEEN DRAWER SIDEBOARD            6              165.00   
          RUSTIC  SEVENTEEN DRAWER SIDEBOARD          5              165.00   
22828     REGENCY MIRROR WITH SHUTTERS                5              145.00   
22823     CHEST NATURAL WOOD 20 DRAWERS               5              110.00   
22929     SCHOOL DESK AND CHAIR                       4               65.00   
22833     HALL CABINET WITH 3 DRAWERS                10               49.95   
22824     3 TIER SWEETHEART GARDEN SHELF              7               35.95   
22803     IVORY EMBROIDERED QUILT                    14               35.75   
22830     UTILTY CABINET WITH HOOKS                  16               26.95   
84968f    SET OF 16 VINTAGE SKY BLUE CUTLERY          1               25.49   
22769     CHALKBOARD KITCHEN ORGANISER               39               24.95   
22782     SET 3 WICKER STORAGE BASKETS               10               24.95   
22764     RUSTIC WOODEN CABINET, GLASS DOORS         22               24.95   
22761     CHEST 7 DRAWER MA CAMPAGNE                 11               24.95   
22783     SET 3 WICKER OVAL BASKETS W LIDS           16               19.95   

                                              Preco_Previsto_Mediano  \
StockCode Description                                                  
22827     GIANT SEVENTEEN DRAWER SIDEBOARD                      3.11   
          RUSTIC  SEVENTEEN DRAWER SIDEBOARD                    1.11   
22828     REGENCY MIRROR WITH SHUTTERS                          7.39   
22823     CHEST NATURAL WOOD 20 DRAWERS                         1.70   
22929     SCHOOL DESK AND CHAIR                                11.61   
22833     HALL CABINET WITH 3 DRAWERS                          48.95   
22824     3 TIER SWEETHEART GARDEN SHELF                       16.46   
22803     IVORY EMBROIDERED QUILT                              23.83   
22830     UTILTY CABINET WITH HOOKS                            58.40   
84968f    SET OF 16 VINTAGE SKY BLUE CUTLERY                   14.77   
22769     CHALKBOARD KITCHEN ORGANISER                         38.59   
22782     SET 3 WICKER STORAGE BASKETS                         17.08   
22764     RUSTIC WOODEN CABINET, GLASS DOORS                   51.97   
22761     CHEST 7 DRAWER MA CAMPAGNE                            8.66   
22783     SET 3 WICKER OVAL BASKETS W LIDS                     14.37   

                                              Erro_Absoluto_Medio  
StockCode Description                                              
22827     GIANT SEVENTEEN DRAWER SIDEBOARD                 155.24  
          RUSTIC  SEVENTEEN DRAWER SIDEBOARD               159.98  
22828     REGENCY MIRROR WITH SHUTTERS                     145.67  
22823     CHEST NATURAL WOOD 20 DRAWERS                    114.40  
22929     SCHOOL DESK AND CHAIR                             53.43  
22833     HALL CABINET WITH 3 DRAWERS                        7.66  
22824     3 TIER SWEETHEART GARDEN SHELF                    19.52  
22803     IVORY EMBROIDERED QUILT                           13.73  
22830     UTILTY CABINET WITH HOOKS                         31.23  
84968f    SET OF 16 VINTAGE SKY BLUE CUTLERY                10.72  
22769     CHALKBOARD KITCHEN ORGANISER                      13.96  
22782     SET 3 WICKER STORAGE BASKETS                       7.66  
22764     RUSTIC WOODEN CABINET, GLASS DOORS                28.57  
22761     CHEST 7 DRAWER MA CAMPAGNE                        15.80  
22783     SET 3 WICKER OVAL BASKETS W LIDS                   5.09

O modelo não está simplesmente errando todos os produtos inéditos da mesma maneira.



Descrições parecidas com produtos baratos podem levar a previsões muito baixas para móveis caros.



Mesmo produtos da mesma categoria podem ter diferenças grandes de preço.



TF-IDF representa semelhanças entre palavras, mas não compreende necessariamente o valor comercial, o tamanho, os materiais ou a qualidade do produto.

In [43]:

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Descrições de treino e teste
desc_treino = (
    treino["Description"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

desc_teste = (
    teste["Description"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Aprender o vocabulário somente com as descrições de treino
vetorizador = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

matriz_treino = vetorizador.fit_transform(desc_treino)
matriz_teste = vetorizador.transform(desc_teste)

# Selecionar somente produtos inéditos
codigos_treino = set(treino["StockCode"].astype(str))
mask_ineditos = ~teste["StockCode"].astype(str).isin(codigos_treino)

# Escolher alguns produtos caros para inspeção
codigos_alvo = ["22827", "22828", "22823"]

for codigo in codigos_alvo:
    posicoes = np.where(
        (teste["StockCode"].astype(str) == codigo).to_numpy()
    )[0]

    if len(posicoes) == 0:
        continue

    pos_teste = posicoes[0]
    similaridades = cosine_similarity(
        matriz_teste[pos_teste],
        matriz_treino
    ).ravel()

    # Os 5 vizinhos textualmente mais semelhantes
    vizinhos = np.argsort(similaridades)[-5:][::-1]

    resultado = treino.iloc[vizinhos][
        ["StockCode", "Description", "Price"]
    ].copy()

    resultado["Similaridade"] = similaridades[vizinhos]

    print(f"\nProduto inédito: {codigo}")
    print("Descrição:", desc_teste.iloc[pos_teste])
    display(resultado.round({"Price": 2, "Similaridade": 3}))



Produto inédito: 22827
Descrição: GIANT SEVENTEEN DRAWER SIDEBOARD


,StockCode,Description,Price,Similaridade
348634,21588,RETROSPOT GIANT TUBE MATCHES,2.55,0.358
154183,84526,GIANT BLACK SUNGLASSES,1.95,0.297
39725,84526,GIANT BLACK SUNGLASSES,1.95,0.297
66524,84526,GIANT BLACK SUNGLASSES,1.95,0.297
172227,84526,GIANT BLACK SUNGLASSES,1.95,0.297



Produto inédito: 22828
Descrição: REGENCY MIRROR WITH SHUTTERS


,StockCode,Description,Price,Similaridade
341403,22423,REGENCY CAKESTAND 3 TIER,12.75,0.277
223902,22423,REGENCY CAKESTAND 3 TIER,12.75,0.277
242593,22423,REGENCY CAKESTAND 3 TIER,12.75,0.277
167570,22423,REGENCY CAKESTAND 3 TIER,12.75,0.277
336531,22423,REGENCY CAKESTAND 3 TIER,12.75,0.277



Produto inédito: 22823
Descrição: CHEST NATURAL WOOD 20 DRAWERS


,StockCode,Description,Price,Similaridade
54242,20791,BLUE TILED SET OF 3 DRAWERS,12.75,0.25
76251,84837,WHITE SWEETHEARTS CABINET 4 DRAWERS,10.75,0.23
17188,84837,WHITE SWEETHEARTS CABINET 4 DRAWERS,12.75,0.23
17743,84837,WHITE SWEETHEARTS CABINET 4 DRAWERS,12.75,0.23
89828,84837,WHITE SWEETHEARTS CABINET 4 DRAWERS,12.75,0.23


Esse resultado melhorou a relevância: agora encontramos produtos que também têm gavetas e armários. Porém, há dois problemas que precisamos corrigir antes de usar essa abordagem no PriceCast.
- Similaridade baixa: o melhor resultado tem apenas 0,25. Isso indica uma correspondência textual limitada, não uma garantia de semelhança comercial.
- Produtos repetidos: o código 84837 aparece quatro vezes. Isso aconteceu porque agrupamos por código e descrição; o mesmo código pode ter descrições diferentes no histórico.
Vamos corrigir o catálogo para ter uma única linha por StockCode, com uma descrição representativa e o preço mediano histórico. Assim, os vizinhos serão produtos distintos de verdade.

In [44]:

print("Dimensões da base:", df_precos.shape)
print("Colunas disponíveis:", df_precos.columns.tolist())

display(df_precos.head())


Dimensões da base: (502514, 9)
Colunas disponíveis: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'AnoMes']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,AnoMes
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12


In [45]:

variaveis = ["treino", "teste"]

for nome in variaveis:
    if nome in globals():
        print(f"{nome}: {globals()[nome].shape}")
    else:
        print(f"{nome}: não encontrado")


treino: (349414, 9)
teste: (153100, 9)


Criar um catálogo com produtos distintos
O objetivo é evitar que o mesmo StockCode apareça várias vezes entre os cinco produtos semelhantes. Para cada código, usaremos o preço mediano observado no treino.

In [46]:

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Criar uma cópia dos dados de treino
base_catalogo = treino.copy()

base_catalogo["StockCode"] = base_catalogo["StockCode"].astype(str)

base_catalogo["Description"] = (
    base_catalogo["Description"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Consolidar o histórico: uma linha por código de produto
catalogo_unico = (
    base_catalogo
    .groupby("StockCode", as_index=False)
    .agg(
        Description=("Description", "first"),
        Price=("Price", "median"),
        Registros=("Price", "count")
    )
)

print("Produtos distintos:", len(catalogo_unico))
print(
    "Códigos duplicados:",
    catalogo_unico["StockCode"].duplicated().sum()
)


Produtos distintos: 4060
Códigos duplicados: 0


In [47]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# 1. Preparar as descrições do catálogo de treino
descricoes_catalogo = (
    catalogo_unico["Description"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 2. Transformar descrições em vetores TF-IDF
vetorizador_catalogo = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)

matriz_catalogo = vetorizador_catalogo.fit_transform(
    descricoes_catalogo
)

# 3. Procurar produtos semelhantes
codigos_alvo = ["22827", "22828", "22823"]

for codigo in codigos_alvo:

    linhas = teste[
        teste["StockCode"].astype(str) == codigo
    ]

    if linhas.empty:
        print(f"\nProduto {codigo} não encontrado no teste.")
        continue

    descricao = (
        linhas["Description"]
        .fillna("")
        .astype(str)
        .iloc[0]
        .upper()
        .strip()
    )

    vetor_produto = vetorizador_catalogo.transform(
        [descricao]
    )

    similaridades = cosine_similarity(
        vetor_produto,
        matriz_catalogo
    ).ravel()

    # Selecionar os cinco produtos mais semelhantes
    indices = np.argsort(similaridades)[::-1][:5]

    resultado = catalogo_unico.iloc[indices][
        ["StockCode", "Description", "Price", "Registros"]
    ].copy()

    resultado["Similaridade"] = similaridades[indices]

    print(f"\nProduto inédito: {codigo}")
    print(f"Descrição: {descricao}")

    display(
        resultado.round({
            "Price": 2,
            "Similaridade": 3
        }).reset_index(drop=True)
    )



Produto inédito: 22827
Descrição: GIANT SEVENTEEN DRAWER SIDEBOARD


,StockCode,Description,Price,Registros,Similaridade
0,84526,GIANT BLACK SUNGLASSES,1.95,30,0.317
1,21589,SWALLOW GIANT TUBE MATCHES,2.55,89,0.273
2,21588,RETRO SPOT GIANT TUBE MATCHES,2.55,103,0.261
3,84972L,GIANT PINK HEART MEMO HOLDER,3.95,46,0.260
4,21681,GIANT MEDINA STAMPED METAL BOWL,9.95,65,0.251



Produto inédito: 22828
Descrição: REGENCY MIRROR WITH SHUTTERS


,StockCode,Description,Price,Registros,Similaridade
0,22423,REGENCY CAKESTAND 3 TIER,12.75,1389,0.309
1,22698,PINK REGENCY TEACUP AND SAUCER,2.95,121,0.264
2,22697,GREEN REGENCY TEACUP AND SAUCER,2.95,142,0.261
3,22699,ROSES REGENCY TEACUP AND SAUCER,2.95,147,0.254
4,20868,SILVER FABRIC MIRROR,1.25,19,0.171



Produto inédito: 22823
Descrição: CHEST NATURAL WOOD 20 DRAWERS


,StockCode,Description,Price,Registros,Similaridade
0,85066,CREAM SWEETHEART MINI CHEST,12.75,327,0.200
1,84687,BEACH HUT SHELF W 3 DRAWERS,5.95,36,0.195
2,20791,BLUE TILED SET OF 3 DRAWERS,12.75,1,0.195
3,84837,WHITE SWEETHEARTS CABINET 4 DRAWERS,12.75,144,0.194
4,22068,BLACK PIRATE TREASURE CHEST,1.65,94,0.185


1. Produtos inéditos com preços mais altos
- Foram identificados 106 produtos inéditos, totalizando 7.110 registros no teste.
- O preço mediano foi £4,95, contra £2,10 para produtos conhecidos.
- Alguns móveis apresentaram preços medianos entre £110 e £165.
Conclusão: os produtos inéditos têm uma distribuição de preços diferente, tornando a previsão mais difícil.

O MAE geral nos produtos inéditos foi £4,30 por registro, e o \(R^2\) foi −0,0366.
Conclusão: o modelo funciona melhor para produtos conhecidos, mas tem dificuldade para estimar preços de produtos inéditos de alto valor.
3. Similaridade textual entre produtos
- O TF-IDF encontrou vizinhos com palavras semelhantes, como GIANT, REGENCY, CHEST e DRAWERS.
- Entretanto, os produtos encontrados nem sempre eram comercialmente comparáveis.
- Por exemplo, um móvel de £165 encontrou como vizinho um produto de óculos de sol por £1,95.
Conclusão: a similaridade textual simples não é suficiente para garantir uma boa previsão de preço.

omparar os dois métodos
Vamos calcular o MAE e o RMSE para:
1. Mediana dos 5 vizinhos: estima o preço usando os cinco produtos mais semelhantes do catálogo de treino.
2. Ridge + TF-IDF: estima o preço usando o modelo que já treinamos.
Antes de executar, confirme que você está no notebook em que já existem as variáveis teste, treino, catalogo_unico, vetorizador_catalogo, matriz_catalogo, modelo_ridge_texto e X_teste_texto.

In [48]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Identificar os produtos que não aparecem no treino
codigos_treino = set(treino["StockCode"].astype(str))

dados_teste = teste.copy()
dados_teste["StockCode"] = dados_teste["StockCode"].astype(str)

ineditos = dados_teste[
    ~dados_teste["StockCode"].isin(codigos_treino)
].copy()

# Previsões do Ridge + TF-IDF
predicoes_ridge = modelo_ridge_texto.predict(X_teste_texto)

dados_teste["Previsto_Ridge_TFIDF"] = predicoes_ridge

# Uma linha por produto inédito
produtos_ineditos = (
    ineditos.groupby("StockCode", as_index=False)
    .agg(
        Description=("Description", "first"),
        Preco_Real=("Price", "median")
    )
)

# Consolidar as previsões do Ridge por produto
ridge_por_produto = (
    dados_teste[
        dados_teste["StockCode"].isin(produtos_ineditos["StockCode"])
    ]
    .groupby("StockCode", as_index=False)
    .agg(Previsto_Ridge_TFIDF=("Previsto_Ridge_TFIDF", "median"))
)

produtos_ineditos = produtos_ineditos.merge(
    ridge_por_produto, on="StockCode", how="left"
)

# Prever pela mediana dos 5 produtos textualmente mais semelhantes
previsoes_vizinhos = []

for _, produto in produtos_ineditos.iterrows():
    descricao = str(produto["Description"]).upper()

    vetor = vetorizador_catalogo.transform([descricao])
    similaridades = (matriz_catalogo @ vetor.T).toarray().ravel()

    # Excluir o próprio código, caso esteja presente no catálogo
    catalogo_filtrado = catalogo_unico[
        catalogo_unico["StockCode"] != produto["StockCode"]
    ].copy()

    indices = catalogo_filtrado.index.to_numpy()
    indices_matriz = catalogo_unico.index.get_indexer(indices)

    ordem = np.argsort(similaridades[indices_matriz])[-5:]
    vizinhos = catalogo_filtrado.iloc[ordem]

    previsoes_vizinhos.append(vizinhos["Price"].median())

produtos_ineditos["Previsto_Vizinhos"] = previsoes_vizinhos

# Comparar os métodos no nível de produto
resultados = []

for coluna, nome in [
    ("Previsto_Vizinhos", "Mediana dos 5 vizinhos"),
    ("Previsto_Ridge_TFIDF", "Ridge + TF-IDF")
]:
    y_real = produtos_ineditos["Preco_Real"]
    y_previsto = produtos_ineditos[coluna]

    resultados.append({
        "Modelo": nome,
        "Produtos": y_previsto.notna().sum(),
        "MAE": mean_absolute_error(y_real, y_previsto),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_previsto))
    })

display(pd.DataFrame(resultados).round(3))

,Modelo,Produtos,MAE,RMSE
0,Mediana dos 5 vizinhos,183,5.618,18.653
1,Ridge + TF-IDF,183,6.371,19.114


A mediana dos cinco vizinhos venceu nesta avaliação.
- MAE: o erro médio absoluto foi aproximadamente £5,62 por produto, contra £6,37 no Ridge + TF-IDF.
- Melhoria no MAE: cerca de 11,8%.
- RMSE: também foi menor com os vizinhos, mas continua elevado por causa de erros grandes em alguns produtos caros.
O RMSE penaliza mais os erros extremos. Por isso, mesmo com um MAE relativamente baixo, alguns produtos podem continuar recebendo previsões muito distantes do preço real.

In [49]:

import numpy as np
import pandas as pd

# Selecionar os 10 produtos inéditos de maior preço real mediano
top_produtos = (
    produtos_ineditos
    .sort_values("Preco_Real", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

detalhes_vizinhos = []

for _, produto in top_produtos.iterrows():
    codigo = str(produto["StockCode"])
    descricao = str(produto["Description"]).upper()

    # Transformar a descrição do produto em vetor TF-IDF
    vetor = vetorizador_catalogo.transform([descricao])
    similaridades = (matriz_catalogo @ vetor.T).toarray().ravel()

    # Excluir o próprio código, se estiver no catálogo
    mascara = catalogo_unico["StockCode"].astype(str).to_numpy() != codigo
    indices_validos = np.flatnonzero(mascara)

    # Selecionar os 5 vizinhos com maior similaridade
    melhores = indices_validos[
        np.argsort(similaridades[indices_validos])[-5:][::-1]
    ]

    vizinhos = catalogo_unico.iloc[melhores].copy()

    for _, vizinho in vizinhos.iterrows():
        detalhes_vizinhos.append({
            "Codigo_Inedito": codigo,
            "Produto_Inedito": descricao,
            "Preco_Real": produto["Preco_Real"],
            "Preco_Ridge_TFIDF": produto["Previsto_Ridge_TFIDF"],
            "Preco_Previsto_Vizinhos": produto["Previsto_Vizinhos"],
            "Vizinho": vizinho["Description"],
            "Preco_Vizinho": vizinho["Price"],
            "Similaridade": similaridades[
                catalogo_unico.index.get_loc(vizinho.name)
            ]
        })

tabela_vizinhos = pd.DataFrame(detalhes_vizinhos)

display(
    tabela_vizinhos
    .sort_values(
        ["Preco_Real", "Similaridade"],
        ascending=[False, False]
    )
    .round({"Preco_Real": 2, "Preco_Ridge_TFIDF": 2,
            "Preco_Previsto_Vizinhos": 2,
            "Preco_Vizinho": 2, "Similaridade": 3})
)


,Codigo_Inedito,Produto_Inedito,Preco_Real,Preco_Ridge_TFIDF,Preco_Previsto_Vizinhos,Vizinho,Preco_Vizinho,Similaridade
0,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.00,3.02,2.55,GIANT BLACK SUNGLASSES,1.95,0.317
1,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.00,3.02,2.55,SWALLOW GIANT TUBE MATCHES,2.55,0.273
2,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.00,3.02,2.55,RETRO SPOT GIANT TUBE MATCHES,2.55,0.261
3,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.00,3.02,2.55,GIANT PINK HEART MEMO HOLDER,3.95,0.260
4,22827,GIANT SEVENTEEN DRAWER SIDEBOARD,165.00,3.02,2.55,GIANT MEDINA STAMPED METAL BOWL,9.95,0.251
5,22828,REGENCY MIRROR WITH SHUTTERS,145.00,7.39,2.95,REGENCY CAKESTAND 3 TIER,12.75,0.309
6,22828,REGENCY MIRROR WITH SHUTTERS,145.00,7.39,2.95,PINK REGENCY TEACUP AND SAUCER,2.95,0.264
7,22828,REGENCY MIRROR WITH SHUTTERS,145.00,7.39,2.95,GREEN REGENCY TEACUP AND SAUCER,2.95,0.261
8,22828,REGENCY MIRROR WITH SHUTTERS,145.00,7.39,2.95,ROSES REGENCY TEACUP AND SAUCER,2.95,0.254
9,22828,REGENCY MIRROR WITH SHUTTERS,145.00,7.39,2.95,SILVER FABRIC MIRROR,1.25,0.171


ão vamos trocar de modelo ainda. Primeiro, vamos testar uma regra simples: não utilizar vizinhos com similaridade muito baixa.
Uma observação importante: um limite mínimo de similaridade, sozinho, não resolve o problema. O exemplo do móvel mostra que até uma similaridade de 0,317 pode ser enganosa. Vamos começar testando um limite de 0,30 e comparar os resultados com o método atual.

In [50]:

# Quantidade de vizinhos que atingem diferentes limites
limites = [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

resumo_limites = []

for limite in limites:
    elegiveis = (
        tabela_vizinhos[tabela_vizinhos["Similaridade"] >= limite]
        .groupby("Codigo_Inedito")["Vizinho"]
        .nunique()
    )

    resumo_limites.append({
        "Similaridade mínima": limite,
        "Produtos com pelo menos 1 vizinho": len(elegiveis),
        "Produtos com 5 vizinhos": int((elegiveis >= 5).sum())
    })

display(pd.DataFrame(resumo_limites))


,Similaridade mínima,Produtos com pelo menos 1 vizinho,Produtos com 5 vizinhos
0,0.20,10,6
1,0.25,9,2
2,0.30,8,0
3,0.35,3,0
4,0.40,3,0
5,0.50,1,0


A tabela indica que a quantidade de produtos semelhantes cai bastante conforme o critério fica mais rigoroso:

Similaridade 0,20: todos os 10 produtos têm ao menos um vizinho; 6 têm cinco ou mais. É o limite com maior cobertura, mas pode aceitar pares menos parecidos.
Similaridade 0,25: 9 produtos ainda têm vizinhos, mas só 2 têm cinco ou mais.
Similaridade 0,30: 8 produtos têm ao menos um vizinho, porém nenhum tem cinco.
A partir de 0,35: só 3 produtos têm vizinhos.
Em 0,50: apenas 1 produto tem algum vizinho.
Interpretação geral: a maioria das semelhanças encontradas parece estar em níveis baixos ou moderados; poucas são muito fortes. Se o objetivo é encontrar opções para a maioria dos produtos, 0,20–0,25 parece mais útil. Se você prefere aceitar menos resultados para exigir maior semelhança, use um limite mais alto. A tabela mostra essa troca entre cobertura e rigor, mas, sozinha, não diz qual limite é o melhor: isso depende de conferir se os vizinhos encontrados fazem sentido para os produtos.